# 📑 Mercury Monthly Billing for Finance Report
 
## 🔗 Links
- **[BRD - Mercury Monthly Billing for Finance Report](https://cepalhellas.atlassian.net/wiki/spaces/DBTC/pages/243924993/BRD+-+Mercury+Monthly+Billing+for+Finance+Report)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket](https://cepalhellas.atlassian.net/browse/DBTC-345)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket](https://cepalhellas.atlassian.net/browse/DBTC-959)**  
  A summary of findings from UAT.
 
---

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
    
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")

        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration

        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    print("mode is shared")
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')
#env_prefix = 'p'
#env_prefix = 'p'

## Input Parameters

**report_args:** it is a json which contains all the appropriate parameters to run the report

`
{   
    "portfolio_name": "Mercury",
    "output_file_prefix": "Mercury_MonthlyBilling_",
    "report_output": "/Volumes/qa-cdh-delivery/internalreports/monthlybilling/mercury/",
    "ad_hoc_date": "YYYY-MM-DD"
}
`

**run_date:** is the date value to run the report. The run_date is empty when it is called by the orchestrator and that means "today"

In [0]:
from typing import Any
import shutil
import os

from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional

import delta  # type: ignore
from pyspark.sql import SparkSession

from io import BytesIO
import pandas as pd
import json

from datetime import datetime
from pytz import timezone

In [0]:
from datetime import datetime
from pytz import timezone
import json

athens_tz = timezone('Europe/Athens')

# --- Get report_args and report_name ---
try:
    report_args = dbutils.widgets.get("report_args").replace("'", '"')
except Exception:
    raise ValueError("Missing required input: report_args")

try:
    report_name = dbutils.widgets.get("report_name")
except Exception:
    raise ValueError("Missing required input: report_name")

# --- Optional run_date widget ---
try:
    run_date_widget = dbutils.widgets.get("run_date")
except:
    run_date_widget = ""

# --- Parse JSON ---
json_data = json.loads(report_args)

required_keys = ["portfolio_name", "output_file_prefix", "report_output"]
for key in required_keys:
    if key not in json_data:
        raise KeyError(f"'{key}' key is missing in the report_args JSON data.")

portfolioName = json_data["portfolio_name"]
output_file_prefix = json_data["output_file_prefix"]
report_output = json_data["report_output"]

# --- Resolve ad_hoc_date vs run_date vs today ---
if "ad_hoc_date" in json_data and json_data["ad_hoc_date"]:
    try:
        ad_hoc_date = datetime.strptime(json_data["ad_hoc_date"], '%Y-%m-%d').date()
        eom_date = ad_hoc_date
    except Exception:
        raise ValueError(f"Invalid ad_hoc_date format: {json_data['ad_hoc_date']} — use YYYY-MM-DD")
else:
    if run_date_widget:
        try:
            run_date = datetime.strptime(run_date_widget, '%Y-%m-%d').date()
        except Exception:
            raise ValueError(f"Invalid run_date format: {run_date_widget} — use YYYY-MM-DD")
    else:
        run_date = datetime.now(athens_tz).date()

    # Use Spark SQL to calculate EOM date (end of previous month) unless an ad_hoc_date is given then eom_date = ad_hoc_date
    spark.sql(f"CREATE OR REPLACE TEMP VIEW tmp_run_date AS SELECT DATE('{run_date}') AS base_date")
    eom_date = spark.sql("SELECT DATE_ADD(DATE_TRUNC('month', base_date), -1) FROM tmp_run_date").collect()[0][0]

# --- Format strings ---
run_date_str = eom_date.strftime('%Y%m%d')
run_date = eom_date.strftime('%Y-%m-%d')
now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

# --- Debug print ---
print(f"Report {report_name} is running with: portfolio_name = {portfolioName}, output_file_prefix = {output_file_prefix}, run_date = {run_date}, now = {now_datetime_str}")

# --- Optional Spark setup for future date refs ---
spark.sql(f"CREATE OR REPLACE TEMP VIEW tmp_date AS SELECT DATE('{eom_date}') AS base_date")


eomDate = eom_date

RefMonth = spark.sql("SELECT DATE(DATE_TRUNC('month', base_date)) AS ref_date FROM tmp_date").collect()[0]['ref_date']



print(eomDate)
print(RefMonth)


## Report Generation

In [0]:


# Define the first and last dates of the selected reporting month
FirstDate = spark.sql(f"""
    SELECT MIN(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}'
""").collect()[0][0]

LastDate = spark.sql(f"""
    SELECT MAX(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}'
""").collect()[0][0]

# Define the last business day of the selected reporting month
LastWorkingDate = spark.sql(f"""
    SELECT MAX(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}' AND IsBusinessDay = 1
""").collect()[0][0]

# Define the second business day of the selected reporting month
SecondWorking = spark.sql(f"""
    SELECT MAX(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}' AND BusinessDayOfMonth = 2
""").collect()[0][0]

# Define the first and second business days of the following month
# FirstWorkingNextMonth is the date that acts as the snapshot date for the report
# using as reference the on-premise logic of EffectiveDate <= @FirstWorkingNextMonth AND @FirstWorkingNextMonth < EndDate
FirstWorkingNextMonth = spark.sql(f"""
    SELECT MIN(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = CAST(DATEADD(MONTH, 1, DATEADD(DAY, 1 - DAY('{RefMonth}'), CAST('{RefMonth}' AS DATE))) AS DATE)
    AND BusinessDayOfMonth = 1
""").collect()[0][0]

SecondWorkingNextMonth = spark.sql(f"""
    SELECT MIN(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = CAST(DATEADD(MONTH, 1, DATEADD(DAY, 1 - DAY('{RefMonth}'), CAST('{RefMonth}' AS DATE))) AS DATE)
    AND BusinessDayOfMonth = 2
""").collect()[0][0]

# Define the third day of the selected reporting month
ThirdDayOfMonth = spark.sql(f"""
    SELECT CAST(DATEADD(DAY, 2, DATEADD(DAY, 1 - DAY('{FirstDate}'), CAST('{FirstDate}' AS DATE))) AS DATE)
""").collect()[0][0]

print(eomDate)
print(RefMonth)

In [0]:
# Create the dexposure temporary view
dexposure = spark.sql(f"""
                      select *
                      from `{env_prefix}-cdh`.derivative.dexposure
                      where BusinessDate = {FirstWorkingNextMonth!r} and sourcesystem='profits'
                      and isactiveflag = true and PortfolioName ilike {portfolioName!r}
                      union
                      select *
                      from `{env_prefix}-cdh`.derivative.dexposure
                      where BusinessDate <= {FirstWorkingNextMonth!r} and sourcesystem='profits'
                      and isactiveflag = false and PortfolioName ilike {portfolioName!r}
                      """)

dexposure.createOrReplaceTempView("dexposure")

# Create the dparty temporary view
dparty = spark.sql(f"""
                     select DISTINCT CDI, CASE WHEN L3869Flag = true THEN 'Yes' ELSE 'No' END AS L3869Flag, TaxRegistrationNumber AS TaxNumber
                     from `{env_prefix}-cdh`.derivative.dparty
                     where BusinessDate = {FirstWorkingNextMonth!r} and sourcesystem='profits'
                     and PortfolioName ilike {portfolioName!r}
                     """)

dparty.createOrReplaceTempView("dparty")

In [0]:
# Snapshot mechanism: One or more files can be uploaded, and if a file with the same name is uploaded, the previous data will be overwritten with the new data. This ensures that the table always holds the most current version of the dataset.
# Incremental mechanism: One or more files can be uploaded, and if a file with the same name is uploaded, we use append to add the new data. The file uploaded should contain only the new records with the appropriate VersionDate as a column.

# Create the LegalActionsMapping temporary view that follows the snapshot mechanism
LegalActionsMapping = spark.sql(f"""
                      select *
                      from `{env_prefix}-ReportInputFiles`.internal.LegalActionsMapping
                      where to_date(IngestionTimestamp, 'd/M/yyyy') = (SELECT to_date(MAX(IngestionTimestamp), 'd/M/yyyy') FROM `{env_prefix}-ReportInputFiles`.internal.LegalActionsMapping)
                      """)

LegalActionsMapping.createOrReplaceTempView("LegalActionsMapping")

# Create the LegalPoolPerimeter temporary view that follows the snapshot mechanism
LegalPoolPerimeter = spark.sql(f"""
                      select *
                      from `{env_prefix}-ReportInputFiles`.internal.LegalPoolPerimeter
                      where to_date(IngestionTimestamp, 'd/M/yyyy') = (SELECT to_date(MAX(IngestionTimestamp), 'd/M/yyyy') FROM `{env_prefix}-ReportInputFiles`.internal.LegalPoolPerimeter)
                      """)

LegalPoolPerimeter.createOrReplaceTempView("LegalPoolPerimeter")

# Create the AndrikopoulosChannelsMapping temporary view that follows the snapshot mechanism
AndrikopoulosChannelsMapping = spark.sql(f"""
                      select *
                      from `{env_prefix}-ReportInputFiles`.internal.AndrikopoulosChannelsMapping
                      where to_date(IngestionTimestamp, 'd/M/yyyy') = (SELECT to_date(MAX(IngestionTimestamp), 'd/M/yyyy') FROM `{env_prefix}-ReportInputFiles`.internal.AndrikopoulosChannelsMapping)
                      """)

AndrikopoulosChannelsMapping.createOrReplaceTempView("AndrikopoulosChannelsMapping")

# Create the AndrikopoulosAutoLoan temporary view that follows the incremental mechanism
AndrikopoulosAutoLoan = spark.sql(f"""
                      select *
                      from `qa-ReportInputFiles`.internal.AndrikopoulosAutoLoan
                      where to_date(IngestionTimestamp, 'd/M/yyyy') = (SELECT to_date(MAX(IngestionTimestamp), 'd/M/yyyy') FROM `qa-ReportInputFiles`.internal.AndrikopoulosAutoLoan)
                      """)

AndrikopoulosAutoLoan.createOrReplaceTempView("AndrikopoulosAutoLoan")

# Create the ServicingFees3rdPartyPayRates temporary view that follows the snapshot mechanism
ServicingFees3rdPartyPayRates = spark.sql(f"""
                      select *
                      from `{env_prefix}-ReportInputFiles`.internal.ServicingFees3rdPartyPayRates
                      where to_date(IngestionTimestamp, 'd/M/yyyy') = (SELECT to_date(MAX(IngestionTimestamp), 'd/M/yyyy') FROM `{env_prefix}-ReportInputFiles`.internal.ServicingFees3rdPartyPayRates)
                      """)

ServicingFees3rdPartyPayRates.createOrReplaceTempView("ServicingFees3rdPartyPayRates")

In [0]:
# Create the Without_Status_Cancelled temporary view that filters CDIs with non-cancelled legal steps after 1 January 2022
Without_Status_Cancelled = spark.sql(f"""
    SELECT DISTINCT LS.*
    FROM `{env_prefix}-cdh`.primary.LegalProcessStep LS
    INNER JOIN dexposure dexp
    ON dexp.CDI = LS.CDI
    WHERE LegalStepStatus <> 'Cancelled'
    AND LegalStepStartDate > '2022-01-01'
    AND LegalStepStartDate < '{FirstWorkingNextMonth}'
    AND LS.BusinessDate = '{FirstWorkingNextMonth}'
""")

Without_Status_Cancelled.createOrReplaceTempView("Without_Status_Cancelled")

# Create the AssumedAsLegal temporary view that contains the list of CDIs that have at least one legal step assumed as legal
AssumedAsLegal = spark.sql(f"""
    SELECT DISTINCT a.CDI
    FROM Without_Status_Cancelled a
    INNER JOIN LegalActionsMapping l
    ON l.LegalStepName = a.LegalStepDefinition
    WHERE l.AssumedAsLegal ILIKE '%Yes%'
""")

AssumedAsLegal.createOrReplaceTempView("AssumedAsLegal")

In [0]:
# Create the temporary views that contain all matched collections from Mercury debtors received within the reporting month, including any collections matched by the 2nd working day of the following month
RegularCollections = spark.sql(f"""
    SELECT a.ExposureKey AS AccountNumber
    , cu.CDI AS DebtorCDI
    , cu.TaxNumber
	  , cu.L3869Flag AS L3869Flag
    , t.ActualDate AS ActualDate
    , CAST(`{env_prefix}-cdh`.derivative.converttoeur(t.Currency, t.Amount, t.BusinessDate) AS FLOAT) AS PaymentAmount
	  , t.CollectionSourceKey AS TransactionNumber
    FROM `{env_prefix}-cdh`.derivative.dcollection t
    INNER JOIN dexposure a
    ON a.ExposureKey = t.ExposureKey
    AND a.PortfolioName = t.PortfolioName
    LEFT JOIN dparty cu
    ON cu.CDI = a.CDI
    WHERE t.PortfolioName ilike {portfolioName!r}
    AND t.BusinessDate = '{FirstWorkingNextMonth}'
    AND FinalReason = 'Matched'
    AND CASE WHEN t.MatchedDate IS NULL THEN '1900-01-01' ELSE t.MatchedDate END <= '{SecondWorkingNextMonth}'
    AND t.ActualDate BETWEEN '{FirstDate}' AND '{LastDate}'
""")

RegularCollections.createOrReplaceTempView("RegularCollections")

AdditionalCollections = spark.sql(f"""
    SELECT a.ExposureKey AS AccountNumber
    , cu.CDI AS DebtorCDI
    , cu.TaxNumber
    , cu.L3869Flag AS L3869Flag
    , t.ActualDate AS ActualDate
    , CAST(`{env_prefix}-cdh`.derivative.converttoeur(t.Currency, t.Amount, t.BusinessDate) AS FLOAT) AS PaymentAmount
    , t.CollectionSourceKey AS TransactionNumber
    FROM `{env_prefix}-cdh`.derivative.dcollection t
    INNER JOIN dexposure a
    ON a.ExposureKey = t.ExposureKey
    AND a.PortfolioName = t.PortfolioName
    LEFT JOIN dparty cu
    ON cu.CDI = a.CDI
    WHERE t.PortfolioName ilike {portfolioName!r}
    AND t.BusinessDate = '{FirstWorkingNextMonth}'
    AND FinalReason = 'Matched'
    AND CASE WHEN t.MatchedDate IS NULL THEN '1900-01-01' ELSE t.MatchedDate END <= '{SecondWorkingNextMonth}'
    AND (CASE WHEN t.MatchedDate IS NOT NULL AND LAST_DAY(t.ActualDate) <> LAST_DAY(t.MatchedDate) THEN t.MatchedDate
              ELSE t.ActualDate END) 
    BETWEEN '{ThirdDayOfMonth}' AND '{LastDate}' AND t.ActualDate < '{FirstDate}'
""")

AdditionalCollections.createOrReplaceTempView("AdditionalCollections")

Collections = spark.sql(f"""SELECT * FROM RegularCollections UNION SELECT * FROM AdditionalCollections""")

Collections.createOrReplaceTempView("Collections")

In [0]:
FinalTbl = spark.sql(f"""SELECT 
		       col.ActualDate
		     , col.AccountNumber
		     , col.DebtorCDI
			 , col.TransactionNumber
		     , col.PaymentAmount
		     , CASE WHEN (col.PaymentAmount LIKE '%-%') THEN 'Χ' ELSE 'Π'                                          	END AS Prosimo
		     , col.L3869Flag AS L3869
		     , CASE WHEN (ls.CDI = col.DebtorCDI)  THEN 'TRUE' ELSE 'FALSE'                                  		END AS HasLegalActions
		     --, CASE WHEN (acm.Account = col.AccountNumber) THEN 'YES' ELSE 'NO'                                    	END AS --AutoLoanPerimeter  
		     --, CASE WHEN (al.CDI = col.DebtorCDI) THEN 'Auto loan legal' ELSE '#N/A'                               	END AS --AutoLoanBasedOnHoist
			 , CASE WHEN (LP.VAT = col.TaxNumber) THEN 'Yes' ELSE 'No'                                             	END AS InLegalPoolPerimeter
			 , col.TaxNumber
		FROM Collections col
		LEFT JOIN AssumedAsLegal ls
		ON ls.CDI = col.DebtorCDI						                                   
		-- LEFT JOIN AndrikopoulosChannelsMapping  acm
        -- ON acm.Account = col.AccountNumber 
		-- LEFT JOIN AndrikopoulosAutoLoan al    
        -- ON al.CDI = col.DebtorCDI 
	    LEFT JOIN LegalPoolPerimeter LP          
		ON LP.VAT = col.TaxNumber""")

FinalTbl.createOrReplaceTempView("FinalTbl")

# ReceivablesClassification = spark.sql(f"""SELECT  
# 		       f.AccountNumber
# 		     , CASE WHEN (f.L3869 = 'No'  AND f.HasLegalActions = 'FALSE' AND f.AutoLoanPerimeter = 'YES'  AND f.AutoLoanBasedOnHoist = 'Auto loan legal')  THEN 'Not L.3869-Auto-Legal'
#                     WHEN (f.L3869 = 'Yes' AND f.HasLegalActions = 'FALSE' AND f.AutoLoanPerimeter = 'YES'  AND f.AutoLoanBasedOnHoist = 'Auto loan legal')  THEN 'L.3869-Auto-Legal'
#                 	WHEN (f.L3869 = 'Yes' AND f.HasLegalActions = 'FALSE' AND f.AutoLoanPerimeter = 'YES')	                                                THEN 'L.3869-Auto-No Legal'
#                     WHEN (f.L3869 = 'Yes' AND f.HasLegalActions = 'TRUE'  AND f.AutoLoanPerimeter = 'NO')                                                   THEN 'L.3869-Not Auto-Legal'
#                     WHEN (f.L3869 = 'Yes' AND f.HasLegalActions = 'FALSE' AND f.AutoLoanPerimeter = 'NO')                                                   THEN 'L.3869-Not Auto-No Legal'
#                     WHEN (f.L3869 = 'No'  AND f.HasLegalActions = 'FALSE' AND f.AutoLoanPerimeter = 'YES')                                                  THEN 'Not L.3869-Auto-No Legal'
#                     WHEN (f.L3869 = 'No'  AND f.HasLegalActions = 'TRUE'  AND f.AutoLoanPerimeter = 'NO')                                                   THEN 'Not L.3869-Not Auto-Legal'
#                     WHEN (f.L3869 = 'No'  AND f.HasLegalActions = 'FALSE' AND f.AutoLoanPerimeter = 'NO')                                                   THEN 'Not L.3869-Not Auto-No Legal'
#                END AS Receivables_Classification
# 		FROM FinalTbl f""")

ReceivablesClassification = spark.sql(f"""
    SELECT  
        f.AccountNumber,
        CASE 
            WHEN f.L3869 = 'Yes' AND f.HasLegalActions = 'TRUE'
                THEN 'L.3869-Not Auto-Legal'

            WHEN f.L3869 = 'Yes' AND f.HasLegalActions = 'FALSE'
                THEN 'L.3869-Not Auto-No Legal'

            WHEN f.L3869 = 'No' AND f.HasLegalActions = 'TRUE'
                THEN 'Not L.3869-Not Auto-Legal'

            WHEN f.L3869 = 'No' AND f.HasLegalActions = 'FALSE'
                THEN 'Not L.3869-Not Auto-No Legal'
        END AS Receivables_Classification
    FROM FinalTbl f
""")

ReceivablesClassification.createOrReplaceTempView("ReceivablesClassification")



ServicingFeesRates = spark.sql(f"""SELECT DISTINCT
		       rc.AccountNumber
			 , rc.Receivables_Classification
		     , CollectionFeeRate
		FROM ServicingFees3rdPartyPayRates    f
		INNER JOIN ReceivablesClassification  rc
		ON f.Category = rc.Receivables_Classification""")

ServicingFeesRates.createOrReplaceTempView("ServicingFeesRates")

# Create the ServicingFees temporary view that contains the data to be exported to an Excel file
# ServicingFees = spark.sql(f"""SELECT DISTINCT  
# 			  ActualDate AS `Ημ/νία`
# 			  ,f.AccountNumber AS `Host account number`
# 			  ,f.DebtorCDI AS CDI
#      		  ,TaxNumber AS `Α.Φ.Μ.`
# 			  ,f.TransactionNumber AS `Αρ. Συναλλαγής`
# 			  ,f.PaymentAmount AS `Ποσό`
# 			  ,f.Prosimo AS `Πρόσημο`
# 			  ,'No' AS UnmatchedPayment
# 			  ,f.L3869 AS `L3869 (Yes/No) for collections fee calculation`
#               ,f.AutoLoanBasedOnHoist AS `Auto loan based on Hoist's file`
#               ,f.AutoLoanPerimeter AS `Auto Loan Perimeter`
# 			  ,f.HasLegalActions AS HasLegalActions
# 			  ,rc.Receivables_Classification AS `Receivables Classification`
# 			  ,rc.CollectionFeeRate AS `Fee Rate`
# 			  ,f.PaymentAmount*rc.CollectionFeeRate AS `Fee Amount`
# 			  ,InLegalPoolPerimeter AS `In Legal Pool Perimeter`
# 		FROM FinalTbl f
# 		INNER JOIN ServicingFeesRates rc
# 		ON f.AccountNumber = rc.AccountNumber""")

# ServicingFees = spark.sql(f"""
#     SELECT DISTINCT  
#           ActualDate AS `Ημ/νία`
#         , f.AccountNumber AS `Host account number`
#         , f.DebtorCDI AS CDI
#         , TaxNumber AS `Α.Φ.Μ.`
#         , f.TransactionNumber AS `Αρ. Συναλλαγής`
#         , f.PaymentAmount AS `Ποσό`
#         , f.Prosimo AS `Πρόσημο`
#         , 'No' AS UnmatchedPayment
#         , f.L3869 AS `L3869 (Yes/No) for collections fee calculation`
#         , f.HasLegalActions AS HasLegalActions
#         , rc.Receivables_Classification AS `Receivables Classification`
#         , rc.CollectionFeeRate AS `Fee Rate`
#         , f.PaymentAmount * rc.CollectionFeeRate AS `Fee Amount`
#         , InLegalPoolPerimeter AS `In Legal Pool Perimeter`
#     FROM FinalTbl f
#     INNER JOIN ServicingFeesRates rc
#         ON f.AccountNumber = rc.AccountNumber
# """)

ServicingFees = spark.sql(f"""
    SELECT DISTINCT  
          ActualDate AS `Ημ/νία`
        , f.AccountNumber AS `Host account number`
        , f.DebtorCDI AS CDI
        , TaxNumber AS `Α.Φ.Μ.`
        , f.TransactionNumber AS `Αρ. Συναλλαγής`
        , f.PaymentAmount AS `Ποσό`
        , f.Prosimo AS `Πρόσημο`
        , 'No' AS UnmatchedPayment
        , f.L3869 AS `L3869 (Yes/No) for collections fee calculation`
        , f.HasLegalActions AS HasLegalActions

        , CASE
              WHEN rc.Receivables_Classification = 'L.3869-Not Auto-No Legal'
                  THEN 'L.3869-No Legal'

              WHEN rc.Receivables_Classification = 'L.3869-Not Auto-Legal'
                  THEN 'L.3869-Legal'

              WHEN rc.Receivables_Classification = 'Not L.3869-Not Auto-No Legal'
                  THEN 'Not L.3869-No Legal'

              WHEN rc.Receivables_Classification = 'Not L.3869-Not Auto-Legal'
                  THEN 'Not L.3869-Legal'

              ELSE rc.Receivables_Classification
          END AS `Receivables Classification`

        , rc.CollectionFeeRate AS `Fee Rate`
        , f.PaymentAmount * rc.CollectionFeeRate AS `Fee Amount`
        , InLegalPoolPerimeter AS `In Legal Pool Perimeter`

    FROM FinalTbl f
    INNER JOIN ServicingFeesRates rc
        ON f.AccountNumber = rc.AccountNumber
""")

ServicingFees.createOrReplaceTempView("ServicingFees")





## Report Export

In [0]:
final_report_df = ServicingFees.toPandas()

final_report_df['Ποσό'] = final_report_df['Ποσό'].apply(lambda x: round(x, 2))
final_report_df['Fee Rate'] = final_report_df['Fee Rate'].apply(lambda x: f"{x:.2%}")
final_report_df['Fee Amount'] = final_report_df['Fee Amount'].apply(lambda x: round(x, 2))

In [0]:
# Install required package
%pip install openpyxl

# Set current timestamp string
now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%d')

# Define the output directory path first
output_dir = f"{report_output}{now_datetime_str}/"
 
# Ensure the output directory exists
dbutils.fs.mkdirs(output_dir)
 
# Now define the full output file path including the filename
output_path = f"{output_dir}{output_file_prefix}{eomDate}.xlsx"
 
# Delete the output path if it already exists
if os.path.exists(output_path):
    os.remove(output_path)
 
# Define the temporary file path
temp_file = f"/tmp/{output_file_prefix}{now_datetime_str}.xlsx"
 
# Write DataFrame to a BytesIO buffer
output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer) as writer:
    final_report_df.to_excel(writer, sheet_name=f"{RefMonth.month}.{RefMonth.year}", index=False)
 
# Save the buffer to the temporary file
with open(temp_file, "wb") as f:
    f.write(output_buffer.getvalue())
 
# Copy the temporary file to the final output path
shutil.copy(temp_file, output_path)
 
print(f"Output file is written successfully: {output_path}")

 